# Day 1 · Section 3: NumPy and PyTorch foundations

**Goal:** use vectors, matrices and tensors to calculate the operations you will meet again in embeddings, attention and retrieval. Each topic pairs a NumPy calculation with its PyTorch equivalent. Then you will assemble a small vector search from those pieces.

This is a standalone student notebook. Run in order. NumPy exercises work on a CPU. Colab normally includes PyTorch; if it is absent locally, the PyTorch cells report a skip until it is installed. No workshop slides are included.

**Sections:** 3.1–3.8 · **Estimated time:** 60–90 minutes. Keep an eye on each shape: most mistakes here are axis mistakes.


## Setup · Versions and device

NumPy arrays run on the CPU. PyTorch tensors can use a CPU or a compatible GPU. The same mathematical operation may have different execution cost; this notebook checks correctness rather than benchmarking speed.


In [ ]:
import numpy as np
print('NumPy',np.__version__)
try:
    import torch
    device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print('PyTorch',torch.__version__,'device',device)
except ImportError:
    torch=None;device=None
    print('PyTorch not installed. In Colab it is normally available; install it locally to run paired cells.')
rng=np.random.default_rng(7)
np.set_printoptions(precision=3,suppress=True)


## 3.1 · Arrays, axes and shapes

An LLM token-ID batch commonly has shape `[batch, sequence]`; its embedding lookup has shape `[batch, sequence, hidden]`. The numbers are sizes, and the axis positions determine what each index means.


In [ ]:
token_ids_np=np.array([[3,1,2,0],[2,3,1,0]],dtype=np.int64)
print('shape:',token_ids_np.shape,'ndim:',token_ids_np.ndim,'dtype:',token_ids_np.dtype)
print('first example:',token_ids_np[0])
print('second token in first example:',token_ids_np[0,1])
assert token_ids_np.shape==(2,4)
assert token_ids_np[0,1]==1
# Exercise: select the final token in each row with slicing.


In [ ]:
embedding_table_np=np.array([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.],[.5,.5,.5]])
embeddings_np=embedding_table_np[token_ids_np]
print('table:',embedding_table_np.shape,'IDs:',token_ids_np.shape,'lookup:',embeddings_np.shape)
assert embeddings_np.shape==(2,4,3)
print('token ID 3 row:',embeddings_np[0,0])
if torch is not None:
    table_t=torch.tensor(embedding_table_np,dtype=torch.float32,device=device)
    ids_t=torch.tensor(token_ids_np,dtype=torch.long,device=device)
    vectors_t=table_t[ids_t]
    print('PyTorch:',tuple(vectors_t.shape),vectors_t.dtype,vectors_t.device)
    assert tuple(vectors_t.shape)==embeddings_np.shape


### Reshape, slicing and transpose

`reshape` changes the view of elements only when the element count is preserved. `transpose` changes axis order. These operations are not interchangeable: `[sequence, hidden]` and `[hidden, sequence]` have different meanings.


In [ ]:
A=np.arange(12).reshape(3,4)
print('A:\n',A)
print('rows 1 onward, columns 0–1:\n',A[1:,:2])
print('transpose:',A.T.shape,'reshape:',A.reshape(2,6).shape)
assert A.size==A.T.size==A.reshape(2,6).size
if torch is not None:
    T=torch.arange(12,device=device).reshape(3,4)
    print('PyTorch transpose:',tuple(T.transpose(0,1).shape),'reshape:',tuple(T.reshape(2,6).shape))
    assert tuple(T.transpose(0,1).shape)==A.T.shape


### Broadcasting

Subtracting one value per row from a score matrix is a common way to make softmax stable. A `[rows, 1]` array broadcasts over columns; a `[rows]` array generally has a different meaning.


In [ ]:
scores_np=np.array([[2.,1.,0.],[1002.,1001.,1000.]])
row_max=scores_np.max(axis=1,keepdims=True)
shifted=scores_np-row_max
print('scores',scores_np.shape,'row maxima',row_max.shape,'shifted:\n',shifted)
assert np.array_equal(shifted[0],shifted[1])
if torch is not None:
    scores_t=torch.tensor(scores_np,dtype=torch.float32,device=device)
    shifted_t=scores_t-scores_t.max(dim=1,keepdim=True).values
    assert np.allclose(shifted_t.cpu().numpy(),shifted)


## 3.2 · Dot product

Multiply corresponding entries, then sum. Equal-length vectors are required. The dot product combines *direction* and *length*, so raw scores from vectors with different magnitudes need care.


In [ ]:
u=np.array([2.,1.,-1.]);v=np.array([1.,3.,2.])
component_products=u*v
print('component products:',component_products,'sum:',component_products.sum(),'np.dot:',np.dot(u,v))
assert np.isclose(np.dot(u,v),component_products.sum())
if torch is not None:
    ut=torch.tensor(u,dtype=torch.float32,device=device)
    vt=torch.tensor(v,dtype=torch.float32,device=device)
    print('torch.dot:',torch.dot(ut,vt).item())
    assert np.isclose(torch.dot(ut,vt).item(),np.dot(u,v))


In [ ]:
anchor=np.array([1.,0.])
for name,other in [('aligned',[2.,0.]),('orthogonal',[0.,3.]),('opposite',[-2.,0.])]:
    print(name,'dot=',anchor@np.array(other))
# Exercise: multiply 'aligned' by 10. Does the angle change? Does the dot product change?


## 3.3 · Magnitude and normalization

The L2 norm is the distance from zero. Dividing a nonzero vector by its norm gives unit length. Zero vectors require an explicit rule; below they remain zero rather than causing division by zero.


In [ ]:
def normalize_np(x,axis=-1,eps=1e-12):
    x=np.asarray(x,dtype=float)
    return x/np.maximum(np.linalg.norm(x,axis=axis,keepdims=True),eps)
example=np.array([3.,4.]);unit=normalize_np(example)
print('length:',np.linalg.norm(example),'unit:',unit,'unit length:',np.linalg.norm(unit))
assert np.isclose(np.linalg.norm(unit),1)
print('zero vector:',normalize_np(np.zeros(2)))
if torch is not None:
    import torch.nn.functional as F
    example_t=torch.tensor(example,dtype=torch.float32,device=device)
    unit_t=F.normalize(example_t,dim=0,eps=1e-12)
    print('torch.linalg.vector_norm:',torch.linalg.vector_norm(example_t).item(),'torch normalized:',unit_t.cpu().numpy())
    assert np.allclose(unit_t.cpu().numpy(),unit)


## 3.4 · Cosine similarity

Cosine similarity divides the dot product by both lengths. If both vectors are already unit-normalized, their cosine similarity equals the dot product. It can be negative: opposite directions have cosine −1.


In [ ]:
def cosine_np(a,b):
    a,b=np.asarray(a,dtype=float),np.asarray(b,dtype=float)
    denom=np.linalg.norm(a)*np.linalg.norm(b)
    return float(a@b/denom) if denom>0 else 0.
q=np.array([1.,1.]);candidates=np.array([[1.,1.],[10.,10.],[1.,0.],[-1.,-1.]])
print('raw dot:',candidates@q)
print('cosine:',[round(cosine_np(q,d),3) for d in candidates])
assert np.isclose(cosine_np(q,candidates[0]),cosine_np(q,candidates[1]))
assert np.isclose(normalize_np(q)@normalize_np(candidates[0]),cosine_np(q,candidates[0]))
if torch is not None:
    q_t=torch.tensor(q,dtype=torch.float32,device=device)
    candidates_t=torch.tensor(candidates,dtype=torch.float32,device=device)
    torch_cos=F.cosine_similarity(q_t.unsqueeze(0),candidates_t,dim=1)
    print('PyTorch cosine:',torch_cos.cpu().numpy())


## 3.5 · Matrix multiplication and transpose

`[M, N] @ [N, P] → [M, P]`. The inner sizes must match. A learned projection transforms every token row using the same weight matrix; attention compares every query row with every key row.


In [ ]:
X=np.arange(12,dtype=float).reshape(3,4)/10
W=np.arange(8,dtype=float).reshape(4,2)/10
Y=X@W
print('X',X.shape,'W',W.shape,'X@W',Y.shape,'first output',Y[0])
assert Y.shape==(3,2)
if torch is not None:
    X_t=torch.tensor(X,dtype=torch.float32,device=device)
    W_t=torch.tensor(W,dtype=torch.float32,device=device)
    Y_t=torch.matmul(X_t,W_t)
    assert np.allclose(Y_t.cpu().numpy(),Y,atol=1e-6)


In [ ]:
Q=np.array([[1.,0.],[0.,1.],[1.,1.]])  # [3 query positions, 2 features]
K=np.array([[1.,0.],[1.,1.],[0.,1.]])  # [3 key positions, 2 features]
score_matrix=Q@K.T
print('Q',Q.shape,'K.T',K.T.shape,'all pair scores',score_matrix.shape)
print(score_matrix)
assert score_matrix[0,1]==Q[0]@K[1]
if torch is not None:
    Q_t=torch.tensor(Q,dtype=torch.float32,device=device)
    K_t=torch.tensor(K,dtype=torch.float32,device=device)
    torch_scores=Q_t@K_t.transpose(-2,-1)
    assert np.array_equal(torch_scores.cpu().numpy(),score_matrix)


### Batched matrix multiplication

A leading batch axis is kept separate. A product of `[B, L, D]` and `[B, D, L]` yields `[B, L, L]`, one score matrix per example. The scaling factor in attention is √D.


In [ ]:
batch_Q=np.stack([Q,2*Q])  # [2,3,2]
batch_K=np.stack([K,K])    # [2,3,2]
batch_scores=batch_Q@batch_K.transpose(0,2,1)/np.sqrt(batch_Q.shape[-1])
print('batched scores:',batch_scores.shape)
assert batch_scores.shape==(2,3,3)
if torch is not None:
    tq=torch.tensor(batch_Q,dtype=torch.float32,device=device)
    tk=torch.tensor(batch_K,dtype=torch.float32,device=device)
    ts=tq@tk.transpose(-2,-1)/np.sqrt(tq.shape[-1])
    assert np.allclose(ts.cpu().numpy(),batch_scores)


## 3.6 · Stable softmax

Exponentiate scores and divide by their sum *along the candidate axis*. Subtracting the maximum score first leaves the probabilities unchanged mathematically and avoids overflow. In attention, normalize across keys for each query row; in token generation, normalize across vocabulary candidates.


In [ ]:
def softmax_np(z,axis=-1):
    z=np.asarray(z,dtype=float)
    shifted=z-np.max(z,axis=axis,keepdims=True)
    e=np.exp(shifted)
    return e/e.sum(axis=axis,keepdims=True)
logits=np.array([1000.,1001.,999.])
probabilities=softmax_np(logits)
print('stable probabilities:',np.round(probabilities,3),'sum:',probabilities.sum())
assert np.isclose(probabilities.sum(),1.)
assert np.allclose(probabilities,softmax_np(np.array([0.,1.,-1.])))
if torch is not None:
    z_t=torch.tensor(logits,dtype=torch.float32,device=device)
    print('torch.softmax:',torch.softmax(z_t,dim=-1).cpu().numpy())
    assert np.allclose(torch.softmax(z_t,dim=-1).cpu().numpy(),probabilities,atol=1e-6)


In [ ]:
attention_weights=softmax_np(batch_scores,axis=-1)
print('weights:',attention_weights.shape,'row sums:\n',attention_weights.sum(axis=-1))
assert np.allclose(attention_weights.sum(axis=-1),1.)
if torch is not None:
    torch_weights=torch.softmax(ts,dim=-1)
    assert np.allclose(torch_weights.cpu().numpy(),attention_weights,atol=1e-6)
# Exercise: try axis=1 instead. Which dimension then sums to one?


## 3.7 · Top-k selection

Top-k returns the best values *and their original indices*. Sorting and selecting does not define the scoring function; it only ranks scores already computed. Ties can have different order across implementations, so do not rely on tie order.


In [ ]:
labels=['battery','reset','warranty','returns','network']
scores=np.array([.41,.12,.88,.67,.21]);k=3
indices=np.argsort(-scores,kind='stable')[:k]
print([(labels[int(i)],float(scores[i])) for i in indices])
assert indices.tolist()==[2,3,0]
if torch is not None:
    values_t,indices_t=torch.topk(torch.tensor(scores,dtype=torch.float32,device=device),k)
    print('torch.topk indices:',indices_t.cpu().tolist(),'values:',values_t.cpu().tolist())
    assert indices_t.cpu().tolist()==indices.tolist()


### Top-k token filtering

Selecting a candidate set is separate from sampling within it. The following cell renormalizes probabilities over only the three highest-logit tokens.


In [ ]:
vocab=['manual','hood','door','app','moon'];logits=np.array([2.2,1.6,1.1,.2,-.4])
keep=np.argsort(-logits)[:3]
filtered_probabilities=softmax_np(logits[keep])
print([(vocab[int(i)],round(float(p),3)) for i,p in zip(keep,filtered_probabilities)])
assert np.isclose(filtered_probabilities.sum(),1.)


## 3.8 · Mini vector search

This is the mathematical core of retrieval: represent each document as a vector, normalize, compute all query–document similarities in one matrix multiplication, rank, and map indices back to source labels. The numbers below are illustrative vectors, not embeddings produced by a language model.


In [ ]:
document_names=['battery charging','display reset','warranty period','return policy','network help']
document_vectors=np.array([[.9,.1,0.],[.1,.8,.1],[0.,.1,.9],[.1,.3,.7],[.3,.5,.2]])
query_names=['How long is the warranty?','How do I reset the display?']
query_vectors=np.array([[0.,0.,1.],[0.,1.,0.]])
docs_unit=normalize_np(document_vectors)
queries_unit=normalize_np(query_vectors)
similarities=queries_unit@docs_unit.T  # [queries, documents]
print('similarity matrix',similarities.shape,'\n',np.round(similarities,3))
ranked=np.argsort(-similarities,axis=1)[:,:2]
for name,row in zip(query_names,ranked):
    print(name,'→',[(document_names[int(i)],round(float(similarities[query_names.index(name),i]),3)) for i in row])
assert ranked[0,0]==2 and ranked[1,0]==1


In [ ]:
if torch is not None:
    import torch.nn.functional as F
    docs_t=F.normalize(torch.tensor(document_vectors,dtype=torch.float32,device=device),dim=-1)
    queries_t=F.normalize(torch.tensor(query_vectors,dtype=torch.float32,device=device),dim=-1)
    similarity_t=queries_t@docs_t.transpose(-2,-1)
    values_t,indices_t=torch.topk(similarity_t,k=2,dim=-1)
    print('PyTorch top-2 indices:',indices_t.cpu().tolist())
    assert np.allclose(similarity_t.cpu().numpy(),similarities,atol=1e-6)
else:print('PyTorch retrieval cell skipped; NumPy retrieval is complete.')


## Checks and experiments

1. Add a sixth document and rerun the vector search. Which shapes change?
2. Multiply a document vector by 20. Compare raw dot-product and cosine rankings.
3. Make one query vector zero. Decide how your retrieval system should handle it rather than silently treating it as meaningful.
4. Create a three-query batch. Verify the score-matrix shape and top-2 source names for every query.
5. Change the softmax axis for the attention matrix. Explain why the result is no longer a distribution over Keys for each Query.
6. On a GPU, compare a NumPy CPU array and a PyTorch CUDA tensor: which can be passed directly to NumPy? Use `.detach().cpu().numpy()` when transferring a tensor for inspection.

**Next:** Section 4 uses real token IDs; Sections 6–8 reuse these matrix operations for RoPE and attention; Day 2 applies the mini-search pattern to document embeddings.

API references: [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) · [PyTorch tensors](https://docs.pytorch.org/docs/stable/tensors.html) · [PyTorch topk](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)
